# Adding a Custom Telescope

This tutorial shows how to add a **new telescope** to `nsb2`. By the end you will be able to:

1. Understand the `Instrument` abstraction and what a telescope needs to provide.
2. Build an `EffectiveApertureInstrument` from a per-pixel *effective aperture response* — the recommended path for most IACT cameras.
3. Define an optical `Bandpass`.
4. Package your telescope as a reusable factory function (like `nsb2.instrument.CTAO.LST1North`).

The example is fully self-contained and uses a small synthetic camera so it runs in a few seconds without any downloads.

In [ ]:
import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from astropy.coordinates import AltAz, EarthLocation, SkyCoord, SkyOffsetFrame
from astropy.time import Time

from nsb2.atmosphere import SingleScatteringAtmosphere
from nsb2.core.instrument import EffectiveApertureInstrument
from nsb2.core.lightpath import DirectPath, ScatteredPath
from nsb2.core.pipeline import Pipeline
from nsb2.core.spectral import Bandpass
from nsb2.emitter import airglow

## 1. The `Instrument` contract

Every telescope in `nsb2` is an `Instrument`. The pipeline only ever talks to a telescope through this small interface:

| Method | Purpose |
| --- | --- |
| `pixel_coords(observation)` | Pixel centre positions as a `SkyCoord` in the observation frame. |
| `pixel_radii()` | Per-pixel search radius (radians) used to gather nearby catalog sources. |
| `fov_range()` | Field of view as `((lon_min, lon_max), (lat_min, lat_max))` in radians. |
| `compute_pixel_weights(field, refs, observation)` | Fill in the instrument response weight (effective area) for each source/pixel pair. |
| `bandpass` | The optical `Bandpass` attribute. |

The base class also provides `eval_grid`, `project_discrete`, and `project_continuous` for free.

You *could* implement all of this from scratch by subclassing `Instrument`, but for a camera whose optics can be described by a **per-pixel effective aperture map** you should use the built-in `EffectiveApertureInstrument`, which implements the whole contract for you.

## 2. The effective-aperture response format

`EffectiveApertureInstrument` is constructed from a single `response` dict (or an `.npz` file that behaves like one) with three arrays:

| Key | Shape | Meaning |
| --- | --- | --- |
| `x` | `(N_pix, G)` | Focal-plane *x* offset (radians) sampled per pixel. |
| `y` | `(N_pix, G)` | Focal-plane *y* offset (radians) sampled per pixel. |
| `values` | `(N_pix, G, G)` | Effective collecting area (m²) at each `(x[i], y[j])` grid node, i.e. `values[p, i, j]`. |

In words: for every pixel `p`, `values[p]` is a 2D map over the pixel's local `(x, y)` grid telling you how much effective aperture (in m²) a photon arriving from that sky offset contributes to the pixel. Integrating the map over solid angle gives the pixel's étendue; the peak value gives the on-axis effective area. This is exactly the format stored in `nsb2/instrument/response/*.npz` for the shipped LST/MST/HESS cameras.

Below we synthesise such a response for a tiny 5×5 camera where each pixel has a Gaussian response peaked at its centre.

In [ ]:
def build_synthetic_response(n_side=5, fov_deg=2.0, grid=15, peak_area_m2=400.0, sigma_frac=0.4):
    """Create an effective-aperture response for an ``n_side`` x ``n_side`` camera.

    Each pixel gets a Gaussian effective-area map peaked at its own centre.
    """
    centres = np.linspace(-fov_deg / 2, fov_deg / 2, n_side)
    cx, cy = (a.ravel() for a in np.meshgrid(centres, centres))

    pitch = np.deg2rad(fov_deg / (n_side - 1))  # angular pixel pitch
    half = pitch / 2 * 1.5  # half-width of each pixel's local grid

    x_arr, y_arr, v_arr = [], [], []
    for x0_deg, y0_deg in zip(cx, cy):
        x0, y0 = np.deg2rad(x0_deg), np.deg2rad(y0_deg)
        x = np.linspace(x0 - half, x0 + half, grid)
        y = np.linspace(y0 - half, y0 + half, grid)
        sigma = half * sigma_frac
        gx = np.exp(-((x - x0) ** 2) / (2 * sigma**2))
        gy = np.exp(-((y - y0) ** 2) / (2 * sigma**2))
        v_arr.append(peak_area_m2 * np.outer(gx, gy))  # (G, G) in m^2
        x_arr.append(x)
        y_arr.append(y)

    return {"x": np.array(x_arr), "y": np.array(y_arr), "values": np.array(v_arr)}


response = build_synthetic_response()
response["x"].shape, response["y"].shape, response["values"].shape

## 3. Defining the optical bandpass

A `Bandpass` maps wavelength to a dimensionless transmission (mirror × filter × PMT quantum efficiency × …). You can construct one directly from arrays, from a CSV file with `Bandpass.from_csv`, or from the SVO Filter Profile Service with `Bandpass.from_SVO`.

Here we build a simple bell-shaped Cherenkov-like response peaking near 400 nm.

In [ ]:
wvl = np.linspace(280, 650, 80) * u.nm
transmission = np.exp(-(((wvl.value - 400) / 90) ** 2)) * 0.4  # peak ~40% throughput
bandpass = Bandpass(wvl, transmission)

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(wvl, bandpass(wvl))
ax.set_xlabel("Wavelength [nm]")
ax.set_ylabel("Transmission")
ax.set_title("Custom telescope bandpass")
fig.tight_layout()

## 4. Building the instrument

With a response and a bandpass, the telescope is a one-liner. `EffectiveApertureInstrument` precomputes pixel centres, radii, the FoV, and the per-pixel étendue on construction.

In [ ]:
my_telescope = EffectiveApertureInstrument(response, bandpass)

print(f"Number of pixels : {my_telescope.n_pixels}")
print(f"FoV (lon, lat)   : {np.rad2deg(my_telescope.fov_range())} deg")
print(f"Pixel radius     : {np.rad2deg(my_telescope.pixel_radii()[0]):.3f} deg")

Let's visualise the camera layout — the pixel grid we just defined:

In [ ]:
pix = my_telescope._pix_pos  # (N_pix, 2) centres in radians
fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(np.rad2deg(pix[:, 0]), np.rad2deg(pix[:, 1]), c="tab:blue")
ax.set_xlabel("x offset [deg]")
ax.set_ylabel("y offset [deg]")
ax.set_aspect("equal")
ax.set_title("Synthetic camera pixel layout")
fig.tight_layout()

## 5. Using the telescope in a full prediction

The new telescope now plugs into a `Pipeline` exactly like the shipped instruments. We add a minimal atmosphere and a diffuse airglow source (which loads from local data, no download needed) and predict the night-sky background.

In [ ]:
# Minimal single-scattering atmosphere
def airmass(Z):
    return 1 / (np.cos(Z) + 0.50572 * (96.07995 - np.rad2deg(Z)) ** (-1.6364))


def tau_rayleigh(lam):
    return 0.00879 * lam.to(u.micron).value ** -4.09


def tau_mie(lam):
    return 0.5 * (lam.to(u.nm).value / 380) ** (-1.0)


def tau_absorption(lam):
    return np.zeros(lam.shape)


atmosphere = SingleScatteringAtmosphere(airmass, tau_rayleigh, tau_mie, tau_absorption, g=0.8)

# A diffuse airglow source (loads from packaged data)
glow = airglow.from_eso_skycalc(87 * u.km, 100)

In [ ]:
location = EarthLocation.from_geodetic(16.5, -23.27, 1.8 * u.km)
obstime = Time("2024-06-15T23:00:00", format="isot", scale="utc")
base = AltAz(obstime=obstime, location=location)
target = SkyCoord(alt=70 * u.deg, az=180 * u.deg, frame=base)
observation = SkyOffsetFrame(origin=target, rotation=0 * u.deg)

model = Pipeline(my_telescope, atmosphere, [glow], paths=[DirectPath(), ScatteredPath(nside=32)])
predictions = model.predict(observation)

for p in predictions:
    med = p.rates.to(u.MHz)[:, 1]
    print(f"{p.source_name:20s} | {p.path_name:14s} | median {med.mean():.2f}")

Each `Prediction` carries per-pixel rates of shape `(N_pix, 3)`, where the last axis holds the *min / median / max* over model realisations. We can display the total predicted rate across the synthetic camera:

In [ ]:
total = sum(p.rates.to(u.MHz)[:, 1].value for p in predictions)

fig, ax = plt.subplots(figsize=(4.5, 4))
sc = ax.scatter(
    np.rad2deg(pix[:, 0]), np.rad2deg(pix[:, 1]), c=total, s=200, marker="h", cmap="viridis"
)
fig.colorbar(sc, ax=ax, label="MHz")
ax.set_xlabel("x offset [deg]")
ax.set_ylabel("y offset [deg]")
ax.set_aspect("equal")
ax.set_title("Total NSB — custom telescope")
fig.tight_layout()

## 6. Packaging as a reusable factory

The shipped telescopes are simply factory functions that load a response `.npz` and a bandpass `.dat`, then return an `EffectiveApertureInstrument` (see `nsb2/instrument/CTAO.py` and `nsb2/instrument/HESS.py`). Follow the same pattern to make your telescope reusable:

1. Save your response with `np.savez("MyTelescope.npz", x=..., y=..., values=...)` under `nsb2/instrument/response/`.
2. Save your bandpass as a CSV under `nsb2/instrument/bandpass/` (a `wvl` column in nm plus one or more transmission columns that get multiplied together).
3. Add a factory function that wires them together.

In [ ]:
# Example factory, mirroring nsb2.instrument.CTAO.LST1North
from nsb2.instrument import BANDPASS_PATH, RESPONSE_PATH  # noqa: E402


def MyTelescope():
    response = np.load(RESPONSE_PATH / "MyTelescope.npz")
    bandpass = Bandpass.from_csv(BANDPASS_PATH / "MyTelescope.dat")
    return EffectiveApertureInstrument(response, bandpass)


# (Uncomment once the data files exist)
# telescope = MyTelescope()

## 7. Going fully custom

If your telescope cannot be described by an effective-aperture map (for example a very different optical design), subclass `nsb2.core.instrument.Instrument` directly and implement the four abstract methods listed in Section 1: `pixel_coords`, `pixel_radii`, `fov_range`, and `compute_pixel_weights`, plus set the `bandpass`, `_pix_pos`, and `_pix_area_sr` attributes. The base class then supplies `eval_grid`, `project_discrete`, and `project_continuous`, so your subclass drops straight into a `Pipeline`.